In [48]:
import dukascopy_python
from dukascopy_python.instruments import INSTRUMENT_FX_MAJORS_EUR_USD
import datetime
import pandas as pd

df = dukascopy_python.fetch(
    INSTRUMENT_FX_MAJORS_EUR_USD,
    dukascopy_python.INTERVAL_MIN_15,
    dukascopy_python.OFFER_SIDE_BID,
    datetime.datetime(2024,1, 1),
    datetime.datetime(2024, 6, 1)
)
df.head()

INFO:DUKASCRIPT:current timestamp :2024-01-30T19:15:00
INFO:DUKASCRIPT:current timestamp :2024-02-28T16:30:00
INFO:DUKASCRIPT:current timestamp :2024-03-28T13:30:00
INFO:DUKASCRIPT:current timestamp :2024-04-26T12:45:00
INFO:DUKASCRIPT:current timestamp :2024-05-27T12:15:00


,open,high,low,close,volume
timestamp,,,,,
2024-01-01 22:00:00+00:00,1.10427,1.10433,1.10423,1.10431,274.55
2024-01-01 22:15:00+00:00,1.10430,1.10434,1.10420,1.10430,143.55
2024-01-01 22:30:00+00:00,1.10431,1.10437,1.10431,1.10433,163.80
2024-01-01 22:45:00+00:00,1.10426,1.10447,1.10423,1.10438,237.60
2024-01-01 23:00:00+00:00,1.10446,1.10446,1.10383,1.10383,723.94


In [49]:
def ATR_POSITION_SIZER(account, risk_percent, multiplier):
    gbpusd_rate = 1.32
    TR1 = df['high'] - df['low']
    TR2 = (df['high'] - df['close'].shift(1)).abs()
    TR3 = (df['low'] - df['close'].shift(1)).abs()
    TR = pd.concat([TR1, TR2, TR3], axis =1).max(axis=1)
    ATR = TR.rolling(window=14).mean()
    df['ATR'] = TR.rolling(window=14).mean()
    stop_price = ATR * multiplier
    stop_distance = stop_price / 0.0001

    risk_budget = account * (risk_percent/100)
    risk_budget_usd = risk_budget * gbpusd_rate
    lots = risk_budget_usd / (stop_distance * 10)

    return lots, stop_distance






In [50]:
result = ATR_POSITION_SIZER(10000, 1, 2)
print(result)


(timestamp
2024-01-01 22:00:00+00:00         NaN
2024-01-01 22:15:00+00:00         NaN
2024-01-01 22:30:00+00:00         NaN
2024-01-01 22:45:00+00:00         NaN
2024-01-01 23:00:00+00:00         NaN
                               ...   
2024-05-31 19:45:00+00:00    1.623902
2024-05-31 20:00:00+00:00    1.661871
2024-05-31 20:15:00+00:00    1.711111
2024-05-31 20:30:00+00:00    1.740113
2024-05-31 20:45:00+00:00    1.664865
Length: 10464, dtype: float64, timestamp
2024-01-01 22:00:00+00:00         NaN
2024-01-01 22:15:00+00:00         NaN
2024-01-01 22:30:00+00:00         NaN
2024-01-01 22:45:00+00:00         NaN
2024-01-01 23:00:00+00:00         NaN
                               ...   
2024-05-31 19:45:00+00:00    8.128571
2024-05-31 20:00:00+00:00    7.942857
2024-05-31 20:15:00+00:00    7.714286
2024-05-31 20:30:00+00:00    7.585714
2024-05-31 20:45:00+00:00    7.928571
Length: 10464, dtype: float64)


In [51]:
df['EMA10'] = df['close'].ewm(span=10, adjust=False).mean()
df['EMA30'] = df['close'].ewm(span=30, adjust=False).mean()

df['signal'] = (df['EMA10'] > df['EMA30']).astype(int)
df['lag_signal'] = df['signal'].shift(1)

df.head(50)

,open,high,low,close,volume,ATR,EMA10,EMA30,signal,lag_signal
timestamp,,,,,,,,,,
2024-01-01 22:00:00+00:00,1.10427,1.10433,1.10423,1.10431,274.55,NaN,1.104310,1.104310,0,NaN
2024-01-01 22:15:00+00:00,1.10430,1.10434,1.10420,1.10430,143.55,NaN,1.104308,1.104309,0,0.0
2024-01-01 22:30:00+00:00,1.10431,1.10437,1.10431,1.10433,163.80,NaN,1.104312,1.104311,1,0.0
2024-01-01 22:45:00+00:00,1.10426,1.10447,1.10423,1.10438,237.60,NaN,1.104324,1.104315,1,1.0
2024-01-01 23:00:00+00:00,1.10446,1.10446,1.10383,1.10383,723.94,NaN,1.104235,1.104284,0,1.0
2024-01-01 23:15:00+00:00,1.10383,1.10383,1.10370,1.10376,756.69,NaN,1.104148,1.104250,0,0.0
2024-01-01 23:30:00+00:00,1.10376,1.10383,1.10361,1.10367,634.39,NaN,1.104061,1.104213,0,0.0
2024-01-01 23:45:00+00:00,1.10367,1.10373,1.10356,1.10365,989.44,NaN,1.103987,1.104176,0,0.0
2024-01-02 00:00:00+00:00,1.10366,1.10380,1.10366,1.10371,773.02,NaN,1.103936,1.104146,0,0.0


In [55]:
in_trade = False
entry_price = None
stop_price = None
current_day = None
daily_r = 0.0
daily_limit = -3.0


trades = []

for i in range(len(df)): 
    row = df.iloc[i]
    signal =(row['lag_signal'])
    price = row['close']
    bar_day = df.index[i].date()
    if bar_day != current_day:
        current_day = bar_day
        daily_r = 0.0

    if not in_trade and signal == 1 and not pd.isna(row['ATR']) and daily_r > daily_limit:
        in_trade = True
        entry_price = price
        atr = row['ATR']
        stop_price = entry_price - (atr * 2)
    elif in_trade == True:
        if row['low'] <= stop_price: 
            r_outcome = -1.0
            trades.append(r_outcome)
            daily_r += r_outcome
            in_trade = False
        elif signal == 0:
            exit_price = price
            profit = exit_price - entry_price
            risk_perunit = entry_price - stop_price
            r_outcome = profit / risk_perunit
            trades.append(r_outcome)
            daily_r += r_outcome
            in_trade = False

        



In [56]:
print('Number of trades:', len(trades))
print('all R outcomes:', trades[:10])

Number of trades: 239
all R outcomes: [-1.0, -1.0, np.float64(-0.14078841512475102), np.float64(2.79123173277682), -1.0, np.float64(-0.43032786885249036), np.float64(-0.09016393442618326), -1.0, np.float64(0.056338028169071346), -1.0]


In [57]:
import numpy as np 

trades_array = np.array(trades)
print('Number of trades: ', len(trades_array))
print('Total R value: ', trades_array.sum())
print('R expectancy: ', trades_array.mean())
print('Win rate: ', (trades_array > 0).mean())


Number of trades:  239
Total R value:  3.157851079154078
R expectancy:  0.01321276602156518
Win rate:  0.30962343096234307
